# 실습 1 - AutoGluon Feature Engineering

**서울 공공자전거(따릉이) 데이터를 활용한 자동 전처리 실습**

머신러닝 모델은 수치형 데이터만 처리 가능하다. 그러나 현실의 데이터에는 '겨울', '휴일' 등의 문자열이나 '2017-12-01' 형태의 날짜가 포함되어 있다. 이처럼 사람이 인식하는 데이터를 모델이 처리 가능한 수치로 변환하는 과정을 **전처리(Feature Engineering)** 라 한다.

본 실습에서는 코드를 직접 실행하며 다음 사항을 확인한다.

1. 원본 데이터에는 문자열과 날짜가 포함되어 있으며, 모델은 이를 직접 처리할 수 없다.
2. AutoGluon은 **단일 명령**으로 이러한 값들을 수치로 변환한다.
3. 날짜는 연·월·일·요일로 분해되고, 범주형 문자열은 정수로 변환된다.
4. 결측치는 임의로 대체하지 않으며, 필요 시 연구자가 직접 개입한다.

#### 노트북 사용법
- 본 문서는 **셀(cell)** 단위로 구성된다. 회색 배경은 실행 가능한 **코드 셀**, 흰 배경은 설명에 해당한다.
- 코드 셀 선택 후 **Shift + Enter** 를 입력하면 해당 셀이 실행되며, 결과가 하단에 출력된다.
- 상단에서부터 순서대로 실행한다.


---
## 0. 환경 준비

프로그램 작성 시 필요한 기능을 매번 직접 구현하지 않고, 기존에 구현된 도구 모음을 사용한다. 이러한 도구 모음을 **라이브러리(library)** 라 한다. 아래 코드는 실습에 필요한 세 가지 라이브러리를 불러온다.

- `pandas` : 표 형태의 데이터를 처리하는 라이브러리. 통상 `pd`로 표기한다.
- `numpy` : 수치 연산에 사용하는 라이브러리. 통상 `np`로 표기한다.
- `AutoMLPipelineFeatureGenerator` : AutoGluon의 전처리를 수행하는 도구.


In [ ]:
# !pip install -U pip
# !pip install -U setuptools wheel
# !pip install autogluon

In [ ]:
import pandas as pd
import numpy as np
from autogluon.features.generators import AutoMLPipelineFeatureGenerator

print("준비 완료")

위 셀 실행 후 `준비 완료`가 출력되면 정상적으로 준비가 완료된 것이다.

`import`는 '불러오기', `as`는 '~라는 이름으로'를 의미한다. 예를 들어 `import pandas as pd`는 pandas를 불러와 pd라는 이름으로 사용함을 뜻한다.


---
## 1. 데이터 불러오기 및 원본 확인

실습에 사용할 따릉이 데이터 파일(`SeoulBikeData.csv`)을 불러온다.

- `pd.read_csv(...)` : CSV 파일(쉼표로 구분된 표 형식 파일)을 불러온다. 불러온 표를 **데이터프레임(DataFrame)** 이라 하며, 본 실습에서는 `df`에 저장한다.
- `encoding="latin-1"` : 파일 인코딩 방식을 지정한다. 본 데이터의 컬럼명에 `°C`(섭씨) 등의 특수문자가 포함되어 있어, 해당 설정이 없을 경우 문자 손상 또는 오류가 발생한다.
- `pd.to_datetime(...)` : 문자열로 저장된 날짜(`01/12/2017`)를 날짜형으로 변환한다. `%d/%m/%Y`는 해당 문자열이 '일/월/년' 순서임을 지정한다.

표에서 가로 방향을 **행(row)**, 세로 방향을 **컬럼(column, 열)** 이라 한다.


In [ ]:
df = pd.read_csv("SeoulBikeData.csv", encoding="latin-1")
df["Date"] = pd.to_datetime(df["Date"], format="%d/%m/%Y")

print("데이터 크기:", df.shape, "(행, 열)")
df.head()

`df.shape`는 데이터의 크기를 (행 수, 열 수) 형태로 반환한다. `df.head()`는 데이터의 **상위 5개 행**을 출력하는 명령이다. 전체 데이터는 8,760행에 달하므로, 상위 일부만 확인하여 형태를 파악한다.

#### 확인 사항

- `Seasons` 컬럼의 `Winter`, `Holiday` 컬럼의 `No Holiday`, `Functioning Day` 컬럼의 `Yes`와 같은 **문자열 값**이 존재한다. 사람은 이를 이해할 수 있으나, 모델은 문자열을 직접 처리할 수 없다.
- 원본 파일의 날짜는 `01/12/2017`이었으나 출력 시 `2017-12-01`로 표시된다. 이는 오류가 아니라 문자열이 날짜형으로 정상 변환되었음을 의미한다. 날짜형은 '연-월-일' 표준 형식으로 표시된다.

컬럼이 14개로 많아 한눈에 확인하기 어려우므로, 주요 컬럼만 선별하여 재확인한다. 아래 코드와 같이 대괄호 `[...]` 안에 컬럼명을 나열하면 해당 컬럼만 조회할 수 있다.


In [ ]:
cols = ["Date", "Rented Bike Count", "Hour", "Temperature(°C)",
        "Seasons", "Holiday", "Functioning Day"]
df[cols].head()

---
## 2. 자동 변환: `fit_transform`

AutoGluon에 데이터를 전달하여 모델이 처리 가능한 형태로 자동 변환한다.

- 우선 예측 대상인 `Rented Bike Count`(대여량) 컬럼을 제외한다. 이 값은 **예측 목표**이므로 전처리 대상에서 제외한다. `df.drop(columns=[...])`은 지정 컬럼을 제거하는 명령이며, 결과를 `X`에 저장한다.
- `AutoMLPipelineFeatureGenerator()`로 전처리 도구를 생성하고, `fit_transform(X)`로 변환을 수행한다. 변환 결과는 `X_transformed`에 저장한다.

`fit_transform`은 데이터를 학습하여(fit) 변환한다(transform)는 의미이며, 이 명령이 전처리의 핵심이다.


In [ ]:
X = df.drop(columns=["Rented Bike Count"])   # 예측 대상(정답) 컬럼을 제외

generator = AutoMLPipelineFeatureGenerator()
X_transformed = generator.fit_transform(X)

print("변환 전 컬럼 수:", X.shape[1])
print("변환 후 컬럼 수:", X_transformed.shape[1])
X_transformed.head()

실행 시 다수의 로그(진행 기록)가 출력된다. 이는 AutoGluon이 수행한 작업을 나타내는 내용이며, 오류가 아니다. 로그 하단에 변환된 표가 출력된다.

#### 확인 사항

- 컬럼 수가 13개에서 17개로 증가하였다. 이는 정보가 확장되었음을 의미한다.
- `Date` 컬럼이 `Date.year`(연), `Date.month`(월), `Date.day`(일), `Date.dayofweek`(요일) 등으로 분해되었다.
- `Seasons`, `Holiday`, `Functioning Day`의 문자열 값이 수치로 변환되었다.
- 온도, 습도 등 기존 수치형 컬럼은 그대로 유지되었다.

> **참고:** `generator`는 `fit_transform`을 1회 실행하면 재사용할 수 없다. 따라서 이후 실험에서는 매번 `AutoMLPipelineFeatureGenerator()`로 도구를 **새로 생성**하여 사용한다.


---
## 3. 날짜 분해 결과 확인

날짜의 분해 결과를 확인하기 위해 날짜 관련 컬럼만 선별하여 조회한다.


In [ ]:
date_cols = ["Date.year", "Date.month", "Date.day", "Date.dayofweek"]
X_transformed[date_cols].head()

#### 날짜를 분해하는 이유

- `Date.dayofweek`는 요일을 나타낸다(0=월요일, 1=화요일, … 6=일요일). 첫 번째 행의 값은 4, 즉 금요일이며, 실제로 2017년 12월 1일은 금요일이다.
- 자전거 대여량은 요일에 따라 상이한 양상을 보인다(평일 출퇴근 수요와 주말 수요의 차이 등). 날짜를 단일 값으로 유지할 경우 모델이 이러한 규칙을 학습하기 어려우나, 요일 등으로 분해할 경우 요일별 수요 차이를 학습할 수 있다.


---
## 4. 결측치(NaN) 처리 방식

**결측치**란 값이 비어 있는 상태를 의미하며, 파이썬에서는 `NaN`(Not a Number)으로 표시된다. 측정 누락이나 무응답 등의 경우에 발생한다.

AutoGluon의 결측치 처리 방식을 확인하기 위해, 첫 번째 행의 기온 값을 임의로 제거한다.

- `df_missing = X.copy()` : 원본 훼손 방지를 위해 데이터를 복사한다.
- `.loc[0, "Temperature(°C)"] = np.nan` : 첫 번째 행(0번 행)의 기온 값을 결측치(`NaN`)로 변경한다.


In [ ]:
df_missing = X.copy()
df_missing.loc[0, "Temperature(°C)"] = np.nan   # 첫 번째 행의 기온을 비움

out_missing = AutoMLPipelineFeatureGenerator().fit_transform(df_missing)
print("첫 번째 행의 기온:", out_missing["Temperature(°C)"].iloc[0])

출력값이 `nan`이면 결측치가 그대로 유지되었음을 의미한다.

#### 결과 해석

- 결측치가 `NaN` 상태로 유지되며, 평균 등의 값으로 임의 대체되지 않는다.
- AutoGluon은 결측치를 각 모델에 그대로 전달하고, 모델별 고유 방식으로 처리하도록 한다. 이를 통해 상이한 관점의 모델이 생성되어 최종 성능 향상에 기여한다.
- 따라서 연구자가 결측치를 사전에 대체할 필요가 없다.

> **연구자의 개입이 필요한 경우:** `999`, `-1` 등 수치로 표기된 결측 코드는 AutoGluon이 자동으로 인식하지 못한다. 예를 들어 '측정 불가'를 999로 표기한 데이터의 경우, 모델은 999를 실제 값으로 인식한다. 이 경우 연구자가 아래와 같이 직접 결측 처리해야 한다.
> ```python
> df["열이름"] = df["열이름"].replace(999, np.nan)
> ```


---
## 5. 연구자의 개입: `Hour` 컬럼의 범주형 지정

`Hour`(시간, 0~23) 컬럼은 수치로 저장되어 있어 AutoGluon은 이를 일반 수치로 처리한다. 그러나 시간대는 크기 비교가 무의미하다. 예를 들어 '23시가 0시보다 23배 크다'는 성립하지 않는다. 즉, 시간대는 수치보다 **범주(구분되는 항목)** 에 가깝다.

이 경우 연구자가 해당 컬럼을 수치가 아닌 범주로 처리하도록 지정할 수 있다. 그 방법이 `.astype("category")`이며, `astype`은 자료형을 변환한다는 의미이다.

먼저 별도 지정이 없을 때 `Hour`가 처리되는 형태를 확인한다.


In [ ]:
out_default = AutoMLPipelineFeatureGenerator().fit_transform(X.copy())
print("기본 처리 시 Hour 타입:", out_default["Hour"].dtype)

출력된 `int64`는 정수(integer), 즉 일반 수치로 처리되었음을 의미한다.

이제 `Hour`를 범주형으로 지정한 뒤 재변환한다.


In [ ]:
df_hour = X.copy()
df_hour["Hour"] = df_hour["Hour"].astype("category")   # 연구자의 개입: 범주형으로 지정

out_hour = AutoMLPipelineFeatureGenerator().fit_transform(df_hour)
print("astype 적용 후 Hour 타입:", out_hour["Hour"].dtype)

#### 결과 해석

- 별도 지정이 없을 경우 `Hour`는 수치(`int`)로 처리된다.
- 범주형으로 지정한 경우 `Hour`는 범주형(`category`)으로 처리된다.
- 이처럼 자동화된 처리에 연구자의 지식을 결합하면 모델 성능을 향상시킬 수 있다. 자동화가 대부분을 수행하되, 데이터의 의미를 이해하는 연구자가 이를 보완한다.


---
## 정리

| 원본 데이터 | AutoGluon의 자동 처리 |
|---|---|
| 수치 (온도 등) | 그대로 유지 |
| 범주형 문자열 (Winter 등) | 정수로 변환 |
| 참/거짓 (Yes / No) | 0 / 1로 변환 |
| 날짜 (2017-12-01) | 연·월·일·요일로 분해 |
| 문자열 (텍스트) | 글자 수·단어 수 등 수치로 변환 |
| 결측치 (NaN) | 대체하지 않고 모델에 전달 |

**요약:** AutoGluon은 문자열·날짜·범주가 혼재된 원본 데이터를 모델이 처리 가능한 수치로 자동 변환한다.

**연구자의 역할:** 수치로 표기된 범주의 지정(`astype`), 수치로 위장된 결측 코드(999 등)의 정리. 데이터의 실질적 의미는 연구자만이 판단 가능하다.


In [ ]:
X = df.drop(columns=["Rented Bike Count"])

# 변환 전 (핵심 열만)
before_cols = ["Date", "Seasons", "Holiday"]
print("===== 변환 전 =====")
print(X[before_cols].head(3))

# 변환
out = AutoMLPipelineFeatureGenerator().fit_transform(X)

# 변환 후 (대응되는 열만)
after_cols = ["Date.year", "Date.month", "Date.day", "Date.dayofweek", "Seasons", "Holiday"]
print("===== 변환 후 =====")
print(out[after_cols].head(3))